In [ ]:
import pandas as pd
from typing import Any

from pendo.client import PendoClient
from pendo.endpoint_factory import get_endpoint
from pendo.endpoints import GuideEndpoint

TARGET_APP_ID = "6271591770816512"


def pull_guides(client: PendoClient, *, app_id: int | None = None) -> list[dict[str, Any]]:
    endpoint = get_endpoint("guides", client)
    assert isinstance(endpoint, GuideEndpoint)

    params: dict[str, Any] = {"expand": "*"}
    if app_id is not None:
        params["appId"] = app_id

    data = endpoint.list_guides(params=params)

    if not isinstance(data, list):
        raise TypeError(f"Expected list from /api/v1/guide, got {type(data)}: {str(data)[:200]}")
    return data


def ms_to_dt(ms):
    if pd.isna(ms) or ms in (None, 0, ""):
        return pd.NaT
    return pd.to_datetime(ms, unit="ms")


def guides_to_df(guides: list[dict[str, Any]], target_app_id: str | None = None) -> pd.DataFrame:
    if target_app_id is None:
        filtered = guides
    else:
        filtered = [g for g in guides if str(g.get("appId")) == str(target_app_id)]

    rows = []
    for g in filtered:
        rows.append({
            "guideId": str(g.get("id")),
            "guideName": g.get("name") or g.get("title"),
            "appId": str(g.get("appId")),
            "state": g.get("state"),
            "publishedAt": g.get("publishedAt"),
            "publishedDt": ms_to_dt(g.get("publishedAt")),
            "showsAfter": g.get("showsAfter"),
            "showsAfterDt": ms_to_dt(g.get("showsAfter")),
            "expiresAfter": g.get("expiresAfter"),
            "expiresAfterDt": ms_to_dt(g.get("expiresAfter")),
            "poll_count": len(g.get("polls") or []),
        })

    return pd.DataFrame(rows)

In [ ]:
guides = pull_guides(client)
df_1r = guides_to_df(guides, TARGET_APP_ID)
print(df_1r.shape)
print(df_1r.columns.tolist())
mask = (
    ((df_1r["showsAfterDt"] >= "2026-01-01") & (df_1r["showsAfterDt"] < "2026-04-01")) |
    ((df_1r["publishedDt"] >= "2026-01-01") & (df_1r["publishedDt"] < "2026-04-01"))
)

df_1r.loc[mask, ["guideId", "guideName", "state", "showsAfterDt", "publishedDt", "poll_count"]]
df_1r[df_1r["guideName"].fillna("").str.contains("ux|experience", case=False, regex=True)][
    ["guideId", "guideName", "state", "showsAfterDt", "publishedDt", "poll_count"]
].sort_values(["showsAfterDt", "publishedDt"])

In [ ]:
len(guides)

In [ ]:
df_all = guides_to_df(guides, None)

display(
    df_all[
        df_all["guideName"].fillna("").str.contains("1r|returns", case=False, regex=True)
    ][["guideId", "guideName", "appId", "state", "showsAfterDt", "publishedDt", "poll_count"]]
    .sort_values(["showsAfterDt", "publishedDt"])
)

In [ ]:
%pip install matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# load your CSV
df = pd.read_csv("../outputs/poll-response-list-1775681471938.csv", encoding="utf-8-sig")

# normalize
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
print(df["Date"].min(), "→", df["Date"].max())

In [ ]:
daily = (
    df.set_index("Date")
      .resample("D")
      .size()
      .rename("responses")
      .reset_index()
)

plt.figure(figsize=(12, 4))
plt.plot(daily["Date"], daily["responses"], marker="o")
plt.title("Daily response volume (1R UX-Lite)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
GAP_DAYS = 10  # tweak if needed

df["gap_days"] = df["Date"].diff().dt.total_seconds() / 86400

df["new_window"] = df["gap_days"].isna() | (df["gap_days"] > GAP_DAYS)

df["window_id"] = df["new_window"].cumsum()

df.head(10)

In [ ]:
windows = (
    df.groupby("window_id")
      .agg(
          start=("Date", "min"),
          end=("Date", "max"),
          responses=("Date", "size"),
          avg_score=("Response", "mean")
      )
      .reset_index()
)

windows

In [ ]:
windows["label"] = (
    "1R UX-Lite | "
    + windows["start"].dt.strftime("%Y-%m-%d")
    + " → "
    + windows["end"].dt.strftime("%Y-%m-%d")
)

windows[["window_id", "label", "responses", "avg_score"]]

In [ ]:
df["guideSessionId_synth"] = (
    "1R_UX_LITE_"
    + df.groupby("window_id")["Date"].transform("min").dt.strftime("%Y%m%d")
)

df.head(20)

In [ ]:
print(windows)

print("\nWindow counts:")
print(df["window_id"].value_counts().sort_index())

In [ ]:
import pandas as pd

path = "/Users/HC28ZB2/Library/CloudStorage/OneDrive-TheHomeDepot/pendo-pipeline/db/guide_sessions.parquet"

df = pd.read_parquet(path)

print("Total rows:", len(df))

ou = df[df["guideId"] == "ytQy-UgzLawZOjj5UG4pmqKH9cQ"][
    ["guideId", "guideSessionId", "reportingStart", "reportingEnd", "responseCount"]
].sort_values("reportingStart")

print("\nOrderUp rows:")
print(ou)